# Deepfake Detection — 1D CNN + Optuna Hyperparameter Optimization
**In-the-Wild dataset**

Author — Claudia Pletka

In [1]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q pytorch-optimizer optuna onnx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.4/287.4 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 93.9 MB/s eta 0:00:00


In [2]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import os
import random
import shutil
from contextlib import redirect_stdout

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, precision_score, recall_score, f1_score
)

from pytorch_optimizer import Ranger

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

import onnx

from google.colab import drive

In [3]:
# ── Cell 3: Device & seed ─────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

def set_seed(seed=22):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(22)

Using device: cuda
NVIDIA A100-SXM4-80GB


In [4]:
# ── Cell 4: Mount Drive & load data ───────────────────────────────────────────
drive.mount('/content/drive')

# Load In-the-Wild v2 features
TRAIN_PATH = ('/content/drive/MyDrive/ITW_Features_train_v2.pkl', 'train')
DEV_PATH   = ('/content/drive/MyDrive/ITW_Features_dev_v2.pkl',   'dev')
EVAL_PATH  = ('/content/drive/MyDrive/ITW_Features_eval_v2.pkl',  'eval')

def load_dataset(path):
    df = pd.read_pickle(path[0])
    print(f'DataFrame loaded for {path[1]} set. '
          f'Total samples: {len(df)}, shape: {df.shape}')
    return df

train_df = load_dataset(TRAIN_PATH)
dev_df   = load_dataset(DEV_PATH)
eval_df  = load_dataset(EVAL_PATH)

print('\nLabel distribution:')
print('Train:', train_df['label'].value_counts().to_dict())
print('Dev:  ', dev_df['label'].value_counts().to_dict())
print('Eval: ', eval_df['label'].value_counts().to_dict())

Mounted at /content/drive
DataFrame loaded for train set. Total samples: 26878, shape: (26878, 5)
DataFrame loaded for dev set. Total samples: 2485, shape: (2485, 5)
DataFrame loaded for eval set. Total samples: 2416, shape: (2416, 5)

Label distribution:
Train: {0: 17135, 1: 9743}
Dev:   {0: 1247, 1: 1238}
Eval:  {0: 1581, 1: 835}


In [5]:
# ── Cell 5: Prepare tensors & normalization stats ─────────────────────────────
def prepare_tensors(df, feature_col='1dcnn_features', label_col='label'):
    X = np.stack(df[feature_col].values).transpose(0, 2, 1).astype(np.float32)
    y = df[label_col].values
    return torch.from_numpy(X), torch.from_numpy(y)

X_train_tr, y_train_tr = prepare_tensors(train_df)
X_dev_tr,   y_dev_tr   = prepare_tensors(dev_df)
X_eval_tr,  y_eval_tr  = prepare_tensors(eval_df)

# Normalization stats (computed on train only)
epsilon = 1e-4
x_means = X_train_tr.mean(dim=(0, 2), keepdim=True)
x_stds  = X_train_tr.std(dim=(0, 2),  keepdim=True) + epsilon

print(f'Train : {X_train_tr.shape}, {y_train_tr.shape}')
print(f'Dev   : {X_dev_tr.shape},   {y_dev_tr.shape}')
print(f'Eval  : {X_eval_tr.shape},  {y_eval_tr.shape}')

Train : torch.Size([26878, 40, 126]), torch.Size([26878])
Dev   : torch.Size([2485, 40, 126]),   torch.Size([2485])
Eval  : torch.Size([2416, 40, 126]),  torch.Size([2416])


In [6]:
# ── Cell 6: Model components ──────────────────────────────────────────────────
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.SiLU(),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y.expand_as(x)


class AttentivePooling(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.attention = nn.Linear(in_channels, 1)

    def forward(self, x):
        x_trans  = x.transpose(1, 2)              # (B, T, C)
        weights  = torch.softmax(self.attention(x_trans), dim=1)  # (B, T, 1)
        pooled   = torch.sum(x_trans * weights, dim=1)            # (B, C)
        return pooled

In [7]:
# ── Cell 7: 1D CNN model (Optuna-parameterised) ───────────────────────────────
class Deepfake_1DCNN(nn.Module):
    """
    Parameters
    ----------
    means, stds   : normalization buffers (from train set)
    out_channels  : number of Conv1d output channels
    kernel_size   : Conv1d kernel size
    dilation      : Conv1d dilation factor
    dropout       : dropout rate in classification head
    fc1_out       : hidden size of the first FC layer

    The pooling strategy is fixed (mean + std + max + attentive),
    so the FC input is always out_channels * 4.
    """
    def __init__(self, means, stds,
                 out_channels=32,
                 kernel_size=5,
                 dilation=4,
                 dropout=0.4,
                 fc1_out=16):
        super().__init__()

        self.register_buffer('means', means.detach().clone().view(1, 40, 1))
        self.register_buffer('stds',  stds.detach().clone().view(1, 40, 1))

        # ── Conv block ─────────────────────────────────────────────────────────
        # padding = dilation * (kernel_size - 1) // 2 keeps output length same
        padding = dilation * (kernel_size - 1) // 2
        self.conv1    = nn.Conv1d(in_channels=40, out_channels=out_channels,
                                  kernel_size=kernel_size, stride=1,
                                  padding=padding, dilation=dilation)
        self.bn1      = nn.BatchNorm1d(out_channels)
        self.silu     = nn.SiLU()
        self.se       = SEBlock(out_channels)
        self.att_pool = AttentivePooling(out_channels)

        # ── Classification head ────────────────────────────────────────────────
        # mean + std + max + attentive = out_channels * 4
        self.fc1      = nn.Linear(out_channels * 4, fc1_out)
        self.dropout  = nn.Dropout(dropout)
        self.fc2      = nn.Linear(fc1_out, 2)

    def forward(self, x):
        # Normalize
        x = (x - self.means) / (self.stds + 1e-7)

        # Feature extraction
        x = self.silu(self.bn1(self.conv1(x)))   # (B, C, T)

        # SE attention
        x = self.se(x)

        # Pooling — 4 strategies concatenated
        mean_p       = torch.mean(x, dim=2)
        std_p        = torch.std(x,  dim=2)
        max_p, _     = torch.max(x,  dim=2)
        att_p        = self.att_pool(x)
        x = torch.cat((mean_p, std_p, max_p, att_p), dim=1)  # (B, C*4)

        # Classification
        x = self.dropout(self.silu(self.fc1(x)))
        return self.fc2(x)

In [8]:
# ── Cell 8: Evaluation function ───────────────────────────────────────────────
def eval_model(model, dl, device, dataset_name='Dataset', show_plots=True):
    """
    Returns (eer, fig).
    Pass show_plots=False during Optuna trials to suppress all output.
    """
    model.eval()
    all_labels, all_preds, all_scores = [], [], []

    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            probs = torch.softmax(model(xb), dim=1)
            preds = torch.argmax(probs, dim=1)
            all_labels.extend(yb.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_scores.extend(probs[:, 0].cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_scores = np.array(all_scores)

    acc = accuracy_score(all_labels, all_preds)
    p   = precision_score(all_labels, all_preds, zero_division=0)
    r   = recall_score(all_labels, all_preds, zero_division=0)
    f1  = f1_score(all_labels, all_preds, zero_division=0)

    fpr, tpr, _ = roc_curve(all_labels, all_scores, pos_label=0)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.absolute(fnr - fpr))] * 100

    if show_plots:
        print(f"\n{'='*10} {dataset_name} Results {'='*10}")
        print(f"EER:       {eer:.4f}%")
        print(f"Accuracy:  {acc:.4f}")
        print(f"F1 Score:  {f1:.4f}")
        print(f"Precision: {p:.4f}")
        print(f"Recall:    {r:.4f}")
        print('\nClassification Report:')
        print(classification_report(all_labels, all_preds,
                                    target_names=['Bonafide', 'Spoof']))

        cm = confusion_matrix(all_labels, all_preds)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Bonafide', 'Spoof'],
                    yticklabels=['Bonafide', 'Spoof'])
        ax.set_title(f'Confusion Matrix: {dataset_name}')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        plt.tight_layout()
        plt.show()
        return eer, fig

    return eer, None

In [9]:
# ── Cell 9: Training loop ─────────────────────────────────────────────────────
def training_loop(epochs, model, loss_fn, opt, train_dl, dev_dl,
                  device, scheduler=None, patience=10, trial=None):
    """
    Returns best dev EER achieved.
    Pass `trial` to enable Optuna pruning.
    """
    best_dev_eer = float('inf')
    epochs_without_improvement = 0
    best_model_state = None

    print(f"{'Epoch':<6} | {'Train Loss':<12} | {'Dev EER':<10} | {'LR':<12}")
    print('-' * 52)

    for epoch in range(epochs):
        # ── Training pass ──────────────────────────────────────────────────────
        model.train()
        running_loss = 0.0

        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            y_pred = model(xb)
            loss   = loss_fn(y_pred, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

            if scheduler:
                scheduler.step()

            running_loss += loss.item()

        # ── Evaluation (silent) ────────────────────────────────────────────────
        current_dev_eer, _ = eval_model(model, dev_dl, device, show_plots=False)
        avg_loss   = running_loss / len(train_dl)
        current_lr = opt.param_groups[0]['lr']

        print(f"[{epoch:02d}]   | {avg_loss:.4f}       | "
              f"{current_dev_eer:.2f}%     | {current_lr:.6f}")

        # ── Optuna: report + prune check ───────────────────────────────────────
        if trial is not None:
            trial.report(current_dev_eer, epoch)
            if trial.should_prune():
                print(f'[!] Trial pruned at epoch {epoch}')
                raise optuna.exceptions.TrialPruned()

        # ── Early stopping ─────────────────────────────────────────────────────
        if current_dev_eer < best_dev_eer:
            best_dev_eer = current_dev_eer
            epochs_without_improvement = 0
            best_model_state = {k: v.cpu().clone()
                                for k, v in model.state_dict().items()}
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f'\n[!] Early stopping triggered at epoch {epoch}.')
                print(f'Best Dev EER: {best_dev_eer:.2f}%')
                break

    if best_model_state:
        model.load_state_dict(best_model_state)
    return best_dev_eer

In [10]:
# ── Cell 10: Optuna objective ─────────────────────────────────────────────────
def make_dataloaders(batch_size):
    train_ds = TensorDataset(X_train_tr, y_train_tr)
    dev_ds   = TensorDataset(X_dev_tr,   y_dev_tr)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          pin_memory=True, num_workers=2)
    dev_dl   = DataLoader(dev_ds,   batch_size=batch_size, shuffle=False,
                          pin_memory=True, num_workers=0)
    return train_dl, dev_dl


def objective(trial):
    # ── Search space ──────────────────────────────────────────────────────────
    batch_size     = trial.suggest_categorical('batch_size',    [128, 256, 512])
    lr             = trial.suggest_float('lr',                  1e-4, 1e-2, log=True)
    weight_decay   = trial.suggest_float('weight_decay',        1e-6, 1e-3, log=True)
    out_channels   = trial.suggest_categorical('out_channels',  [16, 32, 64])
    kernel_size    = trial.suggest_categorical('kernel_size',   [3, 5, 7])
    dilation       = trial.suggest_categorical('dilation',      [1, 2, 4])
    dropout        = trial.suggest_float('dropout',             0.1, 0.5)
    fc1_out        = trial.suggest_categorical('fc1_out',       [16, 32, 64])
    optimizer_name = trial.suggest_categorical('optimizer',     ['Ranger', 'Adam'])
    max_lr_factor  = trial.suggest_float('max_lr_factor',       2.0, 10.0)

    set_seed(22)
    train_dl, dev_dl = make_dataloaders(batch_size)

    model = Deepfake_1DCNN(
        x_means, x_stds,
        out_channels=out_channels,
        kernel_size=kernel_size,
        dilation=dilation,
        dropout=dropout,
        fc1_out=fc1_out,
    ).to(device)

    weights = torch.tensor([1.0, 1.0], dtype=torch.float32).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)

    if optimizer_name == 'Ranger':
        opt = Ranger(model.parameters(), lr=lr, alpha=0.5, k=6,
                     weight_decay=weight_decay)
    else:
        opt = torch.optim.Adam(model.parameters(), lr=lr,
                               weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        opt,
        max_lr=lr * max_lr_factor,
        steps_per_epoch=len(train_dl),
        epochs=50,
        pct_start=0.2,
        div_factor=10,
        final_div_factor=100
    )

    best_eer = training_loop(
        epochs=50,
        model=model,
        loss_fn=loss_fn,
        opt=opt,
        train_dl=train_dl,
        dev_dl=dev_dl,
        device=device,
        scheduler=scheduler,
        patience=10,
        trial=trial,
    )
    return best_eer

In [12]:
# ── Cell 11: Run Optuna study ─────────────
STUDY_DB   = 'sqlite:////content/drive/MyDrive/optuna_1dcnn_itw_v2.db'
STUDY_NAME = '1dcnn_itw_v2'

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STUDY_DB,
    load_if_exists=True,
    direction='minimize',
    sampler=TPESampler(seed=22, n_startup_trials=10),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=10),
)

# Sanity check: warn if resuming a study with prior trials
if len(study.trials) > 0:
    print(f'Resuming existing v2 study with {len(study.trials)} prior trials.')
else:
    print('New trial')

study.optimize(
    objective,
    n_trials=50,
    timeout=3 * 3600,
    gc_after_trial=True,
)

print('\n── Optuna complete ──────────────────────────')
print(f'Best EER   : {study.best_value:.4f}%')
print(f'Best params: {study.best_params}')

[I 2026-05-07 17:39:29,571] Using an existing study with name '1dcnn_itw_v2' instead of creating a new one.


Resuming existing v2 study with 26 prior trials.
Epoch  | Train Loss   | Dev EER    | LR          
----------------------------------------------------
[00]   | 0.6748       | 48.47%     | 0.000566
[01]   | 0.6383       | 26.49%     | 0.000863
[02]   | 0.5384       | 12.52%     | 0.001326
[03]   | 0.3485       | 12.44%     | 0.001909
[04]   | 0.2820       | 11.55%     | 0.002554
[05]   | 0.2655       | 9.37%     | 0.003199
[06]   | 0.2523       | 6.70%     | 0.003780
[07]   | 0.2405       | 5.01%     | 0.004240
[08]   | 0.2318       | 3.63%     | 0.004534
[09]   | 0.2259       | 3.55%     | 0.004633
[10]   | 0.2217       | 3.15%     | 0.004625
[11]   | 0.2187       | 3.15%     | 0.004604
[12]   | 0.2167       | 3.07%     | 0.004568
[13]   | 0.2145       | 2.99%     | 0.004519
[14]   | 0.2132       | 3.39%     | 0.004455
[15]   | 0.2121       | 3.31%     | 0.004379
[16]   | 0.2108       | 3.63%     | 0.004290
[17]   | 0.2096       | 3.72%     | 0.004189
[18]   | 0.2090       | 3.80%    

[I 2026-05-07 17:39:49,738] Trial 26 finished with value: 2.988691437802908 and parameters: {'batch_size': 512, 'lr': 0.0006134922372624525, 'weight_decay': 7.62249292896376e-05, 'out_channels': 64, 'kernel_size': 3, 'dilation': 2, 'dropout': 0.3227131929818622, 'fc1_out': 32, 'optimizer': 'Ranger', 'max_lr_factor': 7.5515721469525365}. Best is trial 26 with value: 2.988691437802908.


[23]   | 0.2064       | 4.04%     | 0.003366

[!] Early stopping triggered at epoch 23.
Best Dev EER: 2.99%
Epoch  | Train Loss   | Dev EER    | LR          
----------------------------------------------------
[00]   | 0.6737       | 48.06%     | 0.000227
[01]   | 0.6366       | 25.44%     | 0.000347
[02]   | 0.5235       | 12.92%     | 0.000532
[03]   | 0.3394       | 12.68%     | 0.000766
[04]   | 0.2857       | 11.63%     | 0.001026
[05]   | 0.2694       | 9.69%     | 0.001285
[06]   | 0.2544       | 6.30%     | 0.001519
[07]   | 0.2429       | 4.44%     | 0.001704
[08]   | 0.2344       | 3.55%     | 0.001822
[09]   | 0.2284       | 3.63%     | 0.001863
[10]   | 0.2237       | 3.55%     | 0.001860
[11]   | 0.2201       | 3.39%     | 0.001851
[12]   | 0.2178       | 3.39%     | 0.001837
[13]   | 0.2157       | 3.15%     | 0.001817
[14]   | 0.2136       | 3.55%     | 0.001792
[15]   | 0.2123       | 3.47%     | 0.001761
[16]   | 0.2115       | 3.55%     | 0.001725
[17]   | 0.2104    

[I 2026-05-07 17:40:17,550] Trial 27 finished with value: 3.150242326332795 and parameters: {'batch_size': 256, 'lr': 0.00024782202264478473, 'weight_decay': 1.367847467474754e-05, 'out_channels': 64, 'kernel_size': 3, 'dilation': 4, 'dropout': 0.37931055801849495, 'fc1_out': 32, 'optimizer': 'Ranger', 'max_lr_factor': 7.515972590105536}. Best is trial 26 with value: 2.988691437802908.


...

[20]   | 0.2138       | 3.39%     | 0.001560

[!] Early stopping triggered at epoch 20.
Best Dev EER: 3.07%
Epoch  | Train Loss   | Dev EER    | LR          
----------------------------------------------------
[00]   | 0.6782       | 54.68%     | 0.000335
[01]   | 0.6557       | 34.98%     | 0.000511
[02]   | 0.6068       | 17.37%     | 0.000785
[03]   | 0.4748       | 12.44%     | 0.001130
[04]   | 0.3216       | 12.12%     | 0.001512
[05]   | 0.2826       | 11.47%     | 0.001893
[06]   | 0.2688       | 10.34%     | 0.002237
[07]   | 0.2567       | 7.67%     | 0.002510
[08]   | 0.2467       | 5.74%     | 0.002683
[09]   | 0.2384       | 4.68%     | 0.002742


[I 2026-05-07 17:51:32,649] Trial 59 pruned. 


[10]   | 0.2324       | 3.80%     | 0.002737
[!] Trial pruned at epoch 10
Epoch  | Train Loss   | Dev EER    | LR          
----------------------------------------------------
[00]   | 0.6746       | 45.96%     | 0.000563
[01]   | 0.6374       | 25.12%     | 0.000859
[02]   | 0.5391       | 12.28%     | 0.001319
[03]   | 0.3506       | 11.79%     | 0.001899
[04]   | 0.2807       | 11.15%     | 0.002541
[05]   | 0.2644       | 7.84%     | 0.003183
[06]   | 0.2514       | 5.57%     | 0.003761
[07]   | 0.2403       | 4.68%     | 0.004219
[08]   | 0.2327       | 3.88%     | 0.004511
[09]   | 0.2274       | 3.72%     | 0.004609
[10]   | 0.2234       | 3.31%     | 0.004602
[11]   | 0.2207       | 3.15%     | 0.004580
[12]   | 0.2187       | 2.99%     | 0.004545
[13]   | 0.2165       | 3.15%     | 0.004496
[14]   | 0.2152       | 3.31%     | 0.004433
[15]   | 0.2142       | 3.15%     | 0.004357
[16]   | 0.2130       | 3.39%     | 0.004268
[17]   | 0.2118       | 3.39%     | 0.004168


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7efbb29d13a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1671, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.12/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 1136, in wait
    ready = selector.select(timeout)
            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/selectors.py", line 415, in select
    fd_event_list = self._selector.poll(timeout)
    

KeyboardInterrupt: 

In [13]:
# ── Cell 12: Final training with best hyperparameters ─────────────────────────
best = study.best_params
print('Training final model with:', best)

EPOCHS = 200

set_seed(22)
train_dl, dev_dl = make_dataloaders(best['batch_size'])
eval_ds  = TensorDataset(X_eval_tr, y_eval_tr)
eval_dl  = DataLoader(eval_ds, batch_size=best['batch_size'],
                      shuffle=False, pin_memory=True, num_workers=0)

model = Deepfake_1DCNN(
    x_means, x_stds,
    out_channels=best['out_channels'],
    kernel_size=best['kernel_size'],
    dilation=best['dilation'],
    dropout=best['dropout'],
    fc1_out=best['fc1_out'],
).to(device)

weights = torch.tensor([1.0, 1.0], dtype=torch.float32).to(device)
loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)

if best['optimizer'] == 'Ranger':
    opt = Ranger(model.parameters(), lr=best['lr'], alpha=0.5, k=6,
                 weight_decay=best['weight_decay'])
else:
    opt = torch.optim.Adam(model.parameters(), lr=best['lr'],
                           weight_decay=best['weight_decay'])

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    opt,
    max_lr=best['lr'] * best['max_lr_factor'],
    steps_per_epoch=len(train_dl),
    epochs=EPOCHS,
    pct_start=0.2,
    div_factor=10,
    final_div_factor=100
)

print(model)

training_loop(
    epochs=EPOCHS,
    model=model,
    loss_fn=loss_fn,
    opt=opt,
    train_dl=train_dl,
    dev_dl=dev_dl,
    device=device,
    scheduler=scheduler,
    patience=15,
    trial=None,
)

Training final model with: {'batch_size': 512, 'lr': 0.0006134922372624525, 'weight_decay': 7.62249292896376e-05, 'out_channels': 64, 'kernel_size': 3, 'dilation': 2, 'dropout': 0.3227131929818622, 'fc1_out': 32, 'optimizer': 'Ranger', 'max_lr_factor': 7.5515721469525365}
Deepfake_1DCNN(
  (conv1): Conv1d(40, 64, kernel_size=(3,), stride=(1,), padding=(2,), dilation=(2,))
  (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (silu): SiLU()
  (se): SEBlock(
    (avg_pool): AdaptiveAvgPool1d(output_size=1)
    (fc): Sequential(
      (0): Linear(in_features=64, out_features=8, bias=False)
      (1): SiLU()
      (2): Linear(in_features=8, out_features=64, bias=False)
      (3): Sigmoid()
    )
  )
  (att_pool): AttentivePooling(
    (attention): Linear(in_features=64, out_features=1, bias=True)
  )
  (fc1): Linear(in_features=256, out_features=32, bias=True)
  (dropout): Dropout(p=0.3227131929818622, inplace=False)
  (fc2): Linear(in_features=32, out_

np.float64(2.988691437802908)

In [14]:
# ── Cell 13: Evaluate on Dev set ──────────────────────────────────────────────
dev_eer, dev_fig = eval_model(model, dev_dl, device, 'Development Set', show_plots=True)


========== Development Set Results ==========
EER:       2.9887%
Accuracy:  0.9775
F1 Score:  0.9772
Precision: 0.9868
Recall:    0.9677

Classification Report:
              precision    recall  f1-score   support

    Bonafide       0.97      0.99      0.98      1247
       Spoof       0.99      0.97      0.98      1238

    accuracy                           0.98      2485
   macro avg       0.98      0.98      0.98      2485
weighted avg       0.98      0.98      0.98      2485



In [15]:
# ── Cell 14: Evaluate on Eval set ─────────────────────────────────────────────
eval_eer, eval_fig = eval_model(model, eval_dl, device, 'Evaluation Set', show_plots=True)

print(f'\nGeneralization gap: {eval_eer - dev_eer:.2f}%')


========== Evaluation Set Results ==========
EER:       5.6287%
Accuracy:  0.9727
F1 Score:  0.9591
Precision: 0.9949
Recall:    0.9257

Classification Report:
              precision    recall  f1-score   support

    Bonafide       0.96      1.00      0.98      1581
       Spoof       0.99      0.93      0.96       835

    accuracy                           0.97      2416
   macro avg       0.98      0.96      0.97      2416
weighted avg       0.97      0.97      0.97      2416


Generalization gap: 2.64%


In [16]:
# ── Cell 15: Save model (v2) ─────────
torch.save(model.state_dict(), '/content/drive/MyDrive/1dcnn_best_itw_v2.pth')
torch.save({'means': x_means, 'stds': x_stds},
           '/content/drive/MyDrive/1dcnn_norm_stats_itw_v2.pth')
print('Model and normalization stats saved to Drive (v2 filenames).')

Model and normalization stats saved to Drive (v2 filenames).


In [17]:
# ── Cell 16: Reload model (v2 — for picking up where you left off) ───────────
study = optuna.load_study(
    study_name='1dcnn_itw_v2',
    storage='sqlite:////content/drive/MyDrive/optuna_1dcnn_itw_v2.db'
)
best = study.best_params
print('Best params:', best)

norm_stats = torch.load('/content/drive/MyDrive/1dcnn_norm_stats_itw_v2.pth')
model = Deepfake_1DCNN(
    norm_stats['means'], norm_stats['stds'],
    out_channels=best['out_channels'],
    kernel_size=best['kernel_size'],
    dilation=best['dilation'],
    dropout=best['dropout'],
    fc1_out=best['fc1_out'],
).to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/1dcnn_best_itw_v2.pth'))
model.eval()
print('v2 model reloaded correctly.')

Best params: {'batch_size': 512, 'lr': 0.0006134922372624525, 'weight_decay': 7.62249292896376e-05, 'out_channels': 64, 'kernel_size': 3, 'dilation': 2, 'dropout': 0.3227131929818622, 'fc1_out': 32, 'optimizer': 'Ranger', 'max_lr_factor': 7.5515721469525365}
v2 model reloaded correctly.
